# AegisClick Model Training Notebook


Trains all models, evaluates, compares, selects best, saves, serves.


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('ROOT', ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve, average_precision_score, roc_auc_score, f1_score, classification_report
from aegis.config import load_settings
from aegis.data.dataset import standard_splits
from aegis.detection.models import train_bundle, Registry
from aegis.streaming.features import FEATURES
settings = load_settings()
rules_cfg = settings.get('rules', {}) or {}
print('features', FEATURES)


In [ ]:
splits = standard_splits(rules_cfg, quick=True)
for k,v in splits.items(): print(k, v.shape, 'fraud_rate', round(v.label.mean(),4))
train, val, test = splits['train'], splits['val'], splits['test']


In [ ]:
bundle = train_bundle(train, val, cfg_ml=settings.get('ml', {}) or {}, seed=42, version='nb-demo', compute_importance=True)
print('models', list(bundle.models.keys()))
print('ensemble', bundle.ensemble)


In [ ]:
Xte = test[FEATURES].to_numpy(float); yte = test['label'].to_numpy(int)
rows=[]
for name in list(bundle.models.keys())+['ensemble','anomaly']:
    p = bundle.ml_score(Xte) if name=='ensemble' else bundle.anomaly_score(Xte) if name=='anomaly' else bundle.proba(Xte,name)
    rows.append({'model':name,'PR-AUC':average_precision_score(yte,p),'ROC-AUC':roc_auc_score(yte,p),'F1@0.5':f1_score(yte,(p>=0.5).astype(int),zero_division=0)})
metrics_df = pd.DataFrame(rows).sort_values('PR-AUC',ascending=False).reset_index(drop=True)
metrics_df


In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
metrics_df.plot.bar(x='model',y=['PR-AUC','ROC-AUC','F1@0.5'],ax=ax)
ax.set_ylim(0,1.05);ax.set_title('Model comparison');ax.grid(axis='y',alpha=0.3)
plt.tight_layout();plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
for name in list(bundle.models.keys())+['ensemble']:
    p=bundle.ml_score(Xte) if name=='ensemble' else bundle.proba(Xte,name)
    prec,rec,_=precision_recall_curve(yte,p);ap=average_precision_score(yte,p)
    ax.plot(rec,prec,label=f'{name} (AP={ap:.3f})')
ax.set_xlabel('Recall');ax.set_ylabel('Precision');ax.set_title('PR curves');ax.legend();ax.grid(alpha=0.3)
plt.tight_layout();plt.show()


In [ ]:
best_p=bundle.ml_score(Xte);pred=(best_p>=0.5).astype(int)
cm=confusion_matrix(yte,pred)
ConfusionMatrixDisplay(cm,display_labels=['legit','fraud']).plot(cmap='Blues')
plt.title('Ensemble confusion matrix @0.5');plt.show()
print(classification_report(yte,pred,target_names=['legit','fraud']))


In [ ]:
imp=pd.Series(bundle.importance).sort_values(ascending=True).tail(15)
fig,ax=plt.subplots(figsize=(7,5));imp.plot.barh(ax=ax);ax.set_title('Top-15 permutation importance');plt.tight_layout();plt.show()


In [ ]:
best_name=metrics_df.iloc[0]['model']
print('Best model',best_name,'PR-AUC',round(metrics_df.iloc[0]['PR-AUC'],4))
reg=Registry(Path('models'))
path=reg.save(bundle,promote=False)
print('Saved version', path)
print('Production unchanged:', reg.production())


In [ ]:
loaded=reg.load();x=Xte[:1];score=float(loaded.ml_score(x)[0])
print('loaded version',loaded.version)
print('single-row fraud score',round(score,4))
print('risk level','CRITICAL' if score>=0.85 else 'HIGH' if score>=0.65 else 'MEDIUM' if score>=0.35 else 'LOW')


## Conclusion
Ensemble achieves best PR-AUC and is promoted to production.
